# Deliverable 4 — Paso 4: Evaluación Completa v2 (FT, Control Base en Mismo Entorno y 20 Nuevas)

Evalúa el sistema multi-agente (`Enrutador` -> `Recuperador` ReAct/E5 -> `Redactor` -> `Verificador`) bajo idéntico entorno T4 y configuración:
1. **FT 75 oficiales** (`Redactor` con QLoRA `Pablox0333/qwen3-4b-redactor-udec-d4`, otros 3 con `disable_adapter()`).
2. **Control Base 75 oficiales (`--solo-base`)** con `disable_adapter()` en **los 4 agentes** en el **mismo entorno** (control experimental limpio).
3. **Set ciego `preguntas_nuevas_20.csv` (20 preguntas inéditas, 0 % solapamiento de unidades con train/val y con D3-75)** tanto en **FT** como en **Control Base (`--solo-base`)**.
4. **Comparativas pareadas Base vs FT** (`--solo-comparativa`) y empaquetado final.

## Paso 0 — Revisar GPU y entorno
En Colab selecciona `Entorno de ejecución -> Cambiar tipo de entorno -> GPU (T4 / L4 / A100)`.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    nombre = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {nombre} (sm_{cap[0]}{cap[1]}) | VRAM: {vram} GB")
else:
    raise RuntimeError("No hay GPU activa. Cambia el entorno a GPU en Colab antes de continuar.")

## Paso 1 — Instalar dependencias
Fijamos `huggingface-hub>=0.30,<2.0` para compatibilidad sin tocar `torch` ni `pandas`.

In [ ]:
%pip install -q -U "transformers>=4.51" peft bitsandbytes accelerate "huggingface-hub>=0.30,<2.0" sentence-transformers
print("Dependencias listas.")

## Paso 2 — Cargar repositorio y verificar archivos (incluyendo `preguntas_nuevas_20.csv`)

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
RAMA_GIT = "deliverable-4"
WORKDIR = Path("/content/d4_eval_work")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", RAMA_GIT, REPO_URL, str(WORKDIR)],
    check=True,
)

D4_DIR = WORKDIR / "Deliverable 4"
D3_AGENTES = WORKDIR / "Deliverable 3" / "Deliverable3" / "agentes"
SCRIPT_EVAL = D4_DIR / "scripts" / "evaluar_d4.py"
SCRIPT_AUDIT = D4_DIR / "scripts" / "generar_auditoria_d4.py"
NUEVAS_20_CSV = D4_DIR / "datos" / "preguntas_nuevas_20.csv"
HF_REPO_ID = "Pablox0333/qwen3-4b-redactor-udec-d4"

for p in (
    SCRIPT_EVAL,
    SCRIPT_AUDIT,
    NUEVAS_20_CSV,
    D3_AGENTES / "corpus.json",
    D3_AGENTES / "test_set_50.csv",
    D3_AGENTES / "test_set_dificil_25.csv",
):
    assert p.is_file(), f"Falta archivo requerido: {p}"

print("Commit clonado:", subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "--short", "HEAD"], text=True).strip())
print("HF_REPO_ID:", HF_REPO_ID)

## Paso 3 — Configurar respaldo en Drive y smoke test de aislamiento LoRA (`congelado_50:13`)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
from google.colab import drive

drive.mount("/content/drive")

REANUDAR_CARPETA = None  # <-- Pega aquí la ruta en Drive si deseas retomar una corrida interrumpida
if REANUDAR_CARPETA:
    OUT_DIR = Path(REANUDAR_CARPETA)
else:
    STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    OUT_DIR = Path(f"/content/drive/MyDrive/D4_eval_runs/run_v2_{STAMP}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Carpeta de salida (Drive):", OUT_DIR)

SMOKE_DIR = OUT_DIR / "_smoke"
cmd_smoke = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(SMOKE_DIR),
    "--qids", "congelado_50:13",
]
print("Ejecutando smoke test:", " ".join(cmd_smoke))
subprocess.run(cmd_smoke, check=True)

tel_smoke = json.loads((SMOKE_DIR / "telemetria_llamadas.json").read_text(encoding="utf-8"))["congelado_50:13"]
for c in tel_smoke:
    print(" ", c["agente"], "| usa_adaptador=", c["usa_adaptador"], "| in=", c["input_tokens"], "| out=", c["output_tokens"], "| s=", c["segundos"])

## Paso 4 — Evaluar las 75 preguntas oficiales con **D4 Fine-tuned (`Redactor` LoRA ON)**
Guarda incrementalmente `resultados_agentes_D4.csv` tras cada pregunta (reanudable).

In [ ]:
import subprocess

cmd_eval_ft = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(OUT_DIR),
]
print("Ejecutando FT 75:", " ".join(cmd_eval_ft))
subprocess.run(cmd_eval_ft, check=True)

## Paso 5 — Evaluar **Control Base en el mismo entorno (`--solo-base`)** sobre las 75 preguntas
Ejecuta exactamente el mismo binario, pesos NF4, índice E5, semillas y sesión Colab con `disable_adapter()` en los 4 agentes.
Guarda `resultados_control_base_D4.csv` (también reanudable por pregunta si Colab desconecta).

In [ ]:
import subprocess

cmd_eval_base = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(OUT_DIR),
    "--solo-base",
    "--output-prefix", "control_base_D4",
]
print("Ejecutando Control Base 75:", " ".join(cmd_eval_base))
subprocess.run(cmd_eval_base, check=True)

## Paso 6 — Evaluar set ciego `preguntas_nuevas_20.csv` (FT y Control Base)
20 preguntas inéditas construidas exclusivamente sobre las `21` unidades reservadas (`0` solapamiento con train/val y con D3-75), evaluadas primero con `Redactor FT` y luego con `--solo-base` (~40 min total).

In [ ]:
import subprocess

cmd_n20_ft = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(OUT_DIR),
    "--preguntas-csv", str(NUEVAS_20_CSV),
    "--nombre-set", "nuevas_20",
    "--output-prefix", "nuevas20_ft_D4",
]
print("Ejecutando Nuevas 20 (FT):", " ".join(cmd_n20_ft))
subprocess.run(cmd_n20_ft, check=True)

cmd_n20_base = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(OUT_DIR),
    "--preguntas-csv", str(NUEVAS_20_CSV),
    "--nombre-set", "nuevas_20",
    "--solo-base",
    "--output-prefix", "nuevas20_base_D4",
]
print("Ejecutando Nuevas 20 (Control Base):", " ".join(cmd_n20_base))
subprocess.run(cmd_n20_base, check=True)

## Paso 7 — Comparativa pareada Base vs FT + Auditoría determinista previa

In [ ]:
import subprocess

COMPARAR_IDS = "congelado_50:13,congelado_50:24,congelado_50:36,dificil_25:H03,dificil_25:H07,dificil_25:H14"
cmd_comp_d3 = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(OUT_DIR),
    "--solo-comparativa",
    "--comparar-ids", COMPARAR_IDS,
]
print("Comparativa 6 casos D3:", " ".join(cmd_comp_d3))
subprocess.run(cmd_comp_d3, check=True)

cmd_comp_n20 = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(OUT_DIR),
    "--preguntas-csv", str(NUEVAS_20_CSV),
    "--nombre-set", "nuevas_20",
    "--solo-comparativa",
    "--comparar-ids", "nuevas_20:N01,nuevas_20:N15,nuevas_20:N19",
    "--output-prefix", "nuevas20",
]
print("Comparativa 3 casos Nuevas 20:", " ".join(cmd_comp_n20))
subprocess.run(cmd_comp_n20, check=True)

for cmd_aud in (
    [
        "python3", str(SCRIPT_AUDIT),
        "--eval-dir", str(OUT_DIR),
        "--resultados-csv", str(OUT_DIR / "resultados_agentes_D4.csv"),
        "--control-csv", str(OUT_DIR / "resultados_control_base_D4.csv"),
    ],
    [
        "python3", str(SCRIPT_AUDIT),
        "--eval-dir", str(OUT_DIR),
        "--resultados-csv", str(OUT_DIR / "resultados_nuevas20_ft_D4.csv"),
        "--control-csv", str(OUT_DIR / "resultados_nuevas20_base_D4.csv"),
        "--prefijo-salida", "nuevas20_ft_D4",
    ],
):
    subprocess.run(cmd_aud, check=True)

## Paso 8 — Empaquetar y descargar resultados completos de evaluación v2

In [ ]:
import zipfile
from google.colab import files

bundle_zip = OUT_DIR / "D4_resultados_eval_v2.zip"
patrones = ("resultados_*.csv", "trazas_*.json", "telemetria_*.json", "configuracion_*.json", "comparativa_*.json", "auditoria_*.json", "casos_*.json")
archivos = sorted({p for pat in patrones for p in OUT_DIR.glob(pat) if p.is_file()})
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in archivos:
        zf.write(f, arcname=f.name)

print(f"Paquete generado ({len(archivos)} archivos):", bundle_zip)
files.download(str(bundle_zip))